In [5]:
import numpy
import scipy
import pandas
import hdmf
import pynwb
import allensdk

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pandas.__version__)
print("HDMF:", hdmf.__version__)
print("PyNWB:", pynwb.__version__)
print("AllenSDK:", allensdk.__version__)

NumPy: 1.23.5
SciPy: 1.10.1
pandas: 1.5.3
HDMF: 3.14.6
PyNWB: 2.8.2
AllenSDK: 2.16.2


In [6]:
from pathlib import Path

from allensdk.brain_observatory.ecephys.ecephys_project_cache import (
    EcephysProjectCache,
)

cache_dir = Path("./data/allen_cache").resolve()
cache_dir.mkdir(parents=True, exist_ok=True)

manifest_path = cache_dir / "manifest.json"

print("Cache directory:", cache_dir)
print("Manifest path:", manifest_path)

cache = EcephysProjectCache.from_warehouse(
    manifest=str(manifest_path)
)

print("Cache created successfully.")

Cache directory: /Users/ricky/Library/Mobile Documents/com~apple~CloudDocs/Intern/ABIC implemented/VisionRecon/data/allen_cache
Manifest path: /Users/ricky/Library/Mobile Documents/com~apple~CloudDocs/Intern/ABIC implemented/VisionRecon/data/allen_cache/manifest.json
Cache created successfully.


In [7]:
import time

print("Loading session metadata...", flush=True)

start = time.perf_counter()

sessions = cache.get_session_table()

elapsed = time.perf_counter() - start

print(f"Loaded {len(sessions)} sessions.")
print(f"Time: {elapsed:.1f} seconds")

display(sessions.head())

Loading session metadata...
Loaded 58 sessions.
Time: 0.6 seconds


,published_at,specimen_id,session_type,age_in_days,sex,full_genotype,unit_count,channel_count,probe_count,ecephys_structure_acronyms
id,,,,,,,,,,
715093703,2019-10-03T00:00:00Z,699733581,brain_observatory_1.1,118.0,M,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,884,2219,6,"[CA1, VISrl, nan, PO, LP, LGd, CA3, DG, VISl, ..."
719161530,2019-10-03T00:00:00Z,703279284,brain_observatory_1.1,122.0,M,Sst-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,755,2214,6,"[TH, Eth, APN, POL, LP, DG, CA1, VISpm, nan, N..."
721123822,2019-10-03T00:00:00Z,707296982,brain_observatory_1.1,125.0,M,Pvalb-IRES-Cre/wt;Ai32(RCL-ChR2(H134R)_EYFP)/wt,444,2229,6,"[MB, SCig, PPT, NOT, DG, CA1, VISam, nan, LP, ..."
732592105,2019-10-03T00:00:00Z,717038288,brain_observatory_1.1,100.0,M,wt/wt,824,1847,5,"[grey, VISpm, nan, VISp, VISl, VISal, VISrl]"
737581020,2019-10-03T00:00:00Z,718643567,brain_observatory_1.1,108.0,M,wt/wt,568,2218,6,"[grey, VISmma, nan, VISpm, VISp, VISl, VISrl]"


In [9]:
#VISp  = V1, VISl  = LM, VISpm = PM
target_areas = {"VISp", "VISl", "VISpm"}

matching_sessions = sessions[
    sessions["ecephys_structure_acronyms"].apply(
        lambda areas: target_areas.issubset(set(areas))
    )
].copy()

print(
    "Sessions containing VISp, VISl and VISpm:",
    len(matching_sessions),
)

print("\nSession protocol counts:")
print(matching_sessions["session_type"].value_counts())

display(
    matching_sessions[
        [
            "session_type",
            "unit_count",
            "ecephys_structure_acronyms",
        ]
    ]
)

Sessions containing VISp, VISl and VISpm: 25

Session protocol counts:
brain_observatory_1.1      15
functional_connectivity    10
Name: session_type, dtype: int64


,session_type,unit_count,ecephys_structure_acronyms
id,,,
715093703,brain_observatory_1.1,884,"[CA1, VISrl, nan, PO, LP, LGd, CA3, DG, VISl, ..."
719161530,brain_observatory_1.1,755,"[TH, Eth, APN, POL, LP, DG, CA1, VISpm, nan, N..."
732592105,brain_observatory_1.1,824,"[grey, VISpm, nan, VISp, VISl, VISal, VISrl]"
737581020,brain_observatory_1.1,568,"[grey, VISmma, nan, VISpm, VISp, VISl, VISrl]"
742951821,brain_observatory_1.1,893,"[VISal, nan, grey, VISl, VISrl, VISp, VISpm, VIS]"
746083955,brain_observatory_1.1,582,"[VPM, TH, LGd, CA3, CA2, CA1, VISal, nan, grey..."
750332458,brain_observatory_1.1,902,"[grey, VISrl, nan, VISal, IntG, IGL, LGd, CA3,..."
750749662,brain_observatory_1.1,761,"[LP, DG, CA1, VISp, nan, LGd, CA3, VISrl, VPM,..."
751348571,brain_observatory_1.1,859,"[grey, LGv, CA3, DG, CA1, VISl, nan, APN, POL,..."


In [10]:
print("Loading unit metadata...", flush=True)

units_metadata = cache.get_units()

print("Number of units in metadata:", len(units_metadata))
print("Columns:")
print(units_metadata.columns.tolist())

Loading unit metadata...
Number of units in metadata: 40010
Columns:
['waveform_PT_ratio', 'waveform_amplitude', 'amplitude_cutoff', 'cumulative_drift', 'd_prime', 'waveform_duration', 'ecephys_channel_id', 'firing_rate', 'waveform_halfwidth', 'isi_violations', 'isolation_distance', 'L_ratio', 'max_drift', 'nn_hit_rate', 'nn_miss_rate', 'presence_ratio', 'waveform_recovery_slope', 'waveform_repolarization_slope', 'silhouette_score', 'snr', 'waveform_spread', 'waveform_velocity_above', 'waveform_velocity_below', 'ecephys_probe_id', 'local_index', 'probe_horizontal_position', 'probe_vertical_position', 'anterior_posterior_ccf_coordinate', 'dorsal_ventral_ccf_coordinate', 'left_right_ccf_coordinate', 'ecephys_structure_id', 'ecephys_structure_acronym', 'ecephys_session_id', 'lfp_sampling_rate', 'name', 'phase', 'sampling_rate', 'has_lfp_data', 'date_of_acquisition', 'published_at', 'specimen_id', 'session_type', 'age_in_days', 'sex', 'genotype']


In [11]:
target_unit_counts = (
    units_metadata[
        units_metadata["ecephys_session_id"].isin(
            matching_sessions.index
        )
        & units_metadata["ecephys_structure_acronym"].isin(
            ["VISp", "VISl", "VISpm"]
        )
    ]
    .groupby(
        [
            "ecephys_session_id",
            "ecephys_structure_acronym",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

for area in ["VISp", "VISl", "VISpm"]:
    if area not in target_unit_counts.columns:
        target_unit_counts[area] = 0

display(target_unit_counts.head())

ecephys_structure_acronym,VISl,VISp,VISpm
ecephys_session_id,,,
715093703,42,60,50
719161530,40,52,18
732592105,40,110,66
737581020,22,40,13
742951821,52,33,23


In [12]:
target_unit_counts["target_total"] = (
    target_unit_counts["VISp"]
    + target_unit_counts["VISl"]
    + target_unit_counts["VISpm"]
)

target_unit_counts["minimum_area_count"] = (
    target_unit_counts[
        ["VISp", "VISl", "VISpm"]
    ].min(axis=1)
)

display(
    target_unit_counts.sort_values(
        "target_total",
        ascending=False,
    )
)

ecephys_structure_acronym,VISl,VISp,VISpm,target_total,minimum_area_count
ecephys_session_id,,,,,
794812542,100,108,65,273,65
732592105,40,110,66,216,40
778240327,62,85,68,215,62
831882777,73,65,72,210,65
816200189,65,61,76,202,61
797828357,58,85,52,195,52
778998620,74,75,45,194,45
839068429,47,85,60,192,47
847657808,76,50,55,181,50


In [13]:
ranking = (
    matching_sessions[
        [
            "session_type",
            "unit_count",
        ]
    ]
    .join(target_unit_counts)
)

ranking = ranking.sort_values(
    [
        "target_total",
        "minimum_area_count",
    ],
    ascending=False,
)

display(ranking)

,session_type,unit_count,VISl,VISp,VISpm,target_total,minimum_area_count
id,,,,,,,
794812542,functional_connectivity,1005,100,108,65,273,65
732592105,brain_observatory_1.1,824,40,110,66,216,40
778240327,functional_connectivity,784,62,85,68,215,62
831882777,functional_connectivity,657,73,65,72,210,65
816200189,functional_connectivity,634,65,61,76,202,61
797828357,brain_observatory_1.1,611,58,85,52,195,52
778998620,functional_connectivity,793,74,75,45,194,45
839068429,functional_connectivity,742,47,85,60,192,47
847657808,functional_connectivity,874,76,50,55,181,50


In [14]:
brain_observatory_ranking = (
    ranking[
        ranking["session_type"]
        == "brain_observatory_1.1"
    ]
    .sort_values(
        [
            "target_total",
            "minimum_area_count",
        ],
        ascending=False,
    )
)

print(
    "Number of matching brain_observatory_1.1 sessions:",
    len(brain_observatory_ranking),
)

display(brain_observatory_ranking)

Number of matching brain_observatory_1.1 sessions: 15


,session_type,unit_count,VISl,VISp,VISpm,target_total,minimum_area_count
id,,,,,,,
732592105,brain_observatory_1.1,824,40,110,66,216,40
797828357,brain_observatory_1.1,611,58,85,52,195,52
755434585,brain_observatory_1.1,650,39,75,62,176,39
756029989,brain_observatory_1.1,684,30,51,90,171,30
791319847,brain_observatory_1.1,555,56,93,17,166,17
760345702,brain_observatory_1.1,501,49,72,44,165,44
715093703,brain_observatory_1.1,884,42,60,50,152,42
746083955,brain_observatory_1.1,582,17,14,115,146,14
754312389,brain_observatory_1.1,502,14,102,22,138,14


In [15]:
number_to_download = 5

top_five = brain_observatory_ranking.head(
    number_to_download
).copy()

selected_session_ids = (
    top_five.index.astype(int).tolist()
)

print("Selected session IDs:")
print(selected_session_ids)

display(top_five)

Selected session IDs:
[732592105, 797828357, 755434585, 756029989, 791319847]


,session_type,unit_count,VISl,VISp,VISpm,target_total,minimum_area_count
id,,,,,,,
732592105,brain_observatory_1.1,824,40,110,66,216,40
797828357,brain_observatory_1.1,611,58,85,52,195,52
755434585,brain_observatory_1.1,650,39,75,62,176,39
756029989,brain_observatory_1.1,684,30,51,90,171,30
791319847,brain_observatory_1.1,555,56,93,17,166,17


In [16]:
import shutil

disk_usage = shutil.disk_usage(cache_dir)

free_gib = disk_usage.free / (1024**3)
used_gib = disk_usage.used / (1024**3)
total_gib = disk_usage.total / (1024**3)

print(f"Total storage: {total_gib:.1f} GiB")
print(f"Used storage: {used_gib:.1f} GiB")
print(f"Free storage: {free_gib:.1f} GiB")

if free_gib < 30:
    print("Warning: download fewer sessions.")
else:
    print("Storage is sufficient for the five-session plan.")

Total storage: 460.4 GiB
Used storage: 366.4 GiB
Free storage: 94.0 GiB
Storage is sufficient for the five-session plan.


In [17]:
import time

test_session_id = selected_session_ids[0]

print(f"Opening session {test_session_id}...", flush=True)

start = time.perf_counter()

test_session = cache.get_session_data(test_session_id)

elapsed_minutes = (time.perf_counter() - start) / 60

print(
    f"Session loaded successfully in "
    f"{elapsed_minutes:.1f} minutes."
)

Opening session 732592105...


Downloading:   0%|          | 0.00/2.91G [00:00<?, ?B/s]

/opt/anaconda3/envs/allen-vc/lib/python3.11/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.1.3 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/opt/anaconda3/envs/allen-vc/lib/python3.11/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.2 because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


Session loaded successfully in 3.9 minutes.


In [21]:
test_units = test_session.units

test_area_counts = (
    test_units[
        test_units["ecephys_structure_acronym"].isin(
            ["VISp", "VISl", "VISpm"]
        )
    ]["ecephys_structure_acronym"]
    .value_counts()
    .reindex(
        ["VISp", "VISl", "VISpm"],
        fill_value=0,
    )
)

print("Target-area unit counts:")
print(test_area_counts)

print(
    "\nNumber of stimulus presentations:",
    len(test_session.stimulus_presentations),
)

Target-area unit counts:
VISp     110
VISl      40
VISpm     66
Name: ecephys_structure_acronym, dtype: int64

Number of stimulus presentations: 70388


In [22]:
stimulus_counts = (
    test_session.stimulus_presentations[
        "stimulus_name"
    ]
    .value_counts()
)

display(stimulus_counts)

natural_movie_three    36000
natural_movie_one      18000
static_gratings         6000
natural_scenes          5950
gabors                  3645
drifting_gratings        628
flashes                  150
spontaneous               15
Name: stimulus_name, dtype: int64

In [24]:
import pandas as pd
import shutil
import time
from pathlib import Path

download_log_path = Path(
    "./data/session_download_log.csv"
).resolve()

download_results = []

for position, session_id in enumerate(
    selected_session_ids,
    start=1,
):
    free_before = (
        shutil.disk_usage(cache_dir).free
        / (1024**3)
    )

    print(
        f"\n[{position}/{len(selected_session_ids)}] "
        f"Loading session {session_id}",
        flush=True,
    )

    print(
        f"Free space before: {free_before:.1f} GiB",
        flush=True,
    )

    if free_before < 30:
        print(
            "Stopping because free space is below 30 GiB."
        )
        break

    start = time.perf_counter()

    try:
        current_session = cache.get_session_data(
            session_id
        )

        current_counts = (
            current_session.units[
                current_session.units[
                    "ecephys_structure_acronym"
                ].isin(
                    ["VISp", "VISl", "VISpm"]
                )
            ]["ecephys_structure_acronym"]
            .value_counts()
            .reindex(
                ["VISp", "VISl", "VISpm"],
                fill_value=0,
            )
        )

        stimulus_counts = (
            current_session.stimulus_presentations[
                "stimulus_name"
            ]
            .value_counts()
        )

        free_after = (
            shutil.disk_usage(cache_dir).free
            / (1024**3)
        )

        result = {
            "session_id": session_id,
            "status": "success",
            "minutes": round(
                (time.perf_counter() - start) / 60,
                2,
            ),
            "VISp": int(current_counts["VISp"]),
            "VISl": int(current_counts["VISl"]),
            "VISpm": int(current_counts["VISpm"]),
            "natural_scenes": int(
                stimulus_counts.get("natural_scenes", 0)
            ),
            "free_space_after_GiB": round(
                free_after,
                1,
            ),
        }

    except Exception as error:
        result = {
            "session_id": session_id,
            "status": "failed",
            "error": repr(error),
        }

    download_results.append(result)

    pd.DataFrame(download_results).to_csv(
        download_log_path,
        index=False,
    )

    print(result, flush=True)

print("\nDownload log:")
print(download_log_path)

display(pd.DataFrame(download_results))


[1/5] Loading session 732592105
Free space before: 88.3 GiB
{'session_id': 732592105, 'status': 'success', 'minutes': 0.28, 'VISp': 110, 'VISl': 40, 'VISpm': 66, 'natural_scenes': 5950, 'free_space_after_GiB': 87.3}

[2/5] Loading session 797828357
Free space before: 87.3 GiB


Downloading:   0%|          | 0.00/2.55G [00:00<?, ?B/s]

{'session_id': 797828357, 'status': 'success', 'minutes': 3.42, 'VISp': 85, 'VISl': 58, 'VISpm': 52, 'natural_scenes': 5950, 'free_space_after_GiB': 86.0}

[3/5] Loading session 755434585
Free space before: 86.0 GiB


Downloading:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

{'session_id': 755434585, 'status': 'success', 'minutes': 2.94, 'VISp': 75, 'VISl': 39, 'VISpm': 62, 'natural_scenes': 5950, 'free_space_after_GiB': 83.8}

[4/5] Loading session 756029989
Free space before: 83.8 GiB


Downloading:   0%|          | 0.00/2.58G [00:00<?, ?B/s]

{'session_id': 756029989, 'status': 'success', 'minutes': 3.36, 'VISp': 51, 'VISl': 30, 'VISpm': 90, 'natural_scenes': 5950, 'free_space_after_GiB': 81.4}

[5/5] Loading session 791319847
Free space before: 81.4 GiB


Downloading:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

{'session_id': 791319847, 'status': 'success', 'minutes': 2.87, 'VISp': 93, 'VISl': 56, 'VISpm': 17, 'natural_scenes': 5950, 'free_space_after_GiB': 79.3}

Download log:
/Users/ricky/Library/Mobile Documents/com~apple~CloudDocs/Intern/ABIC implemented/VisionRecon/data/session_download_log.csv


,session_id,status,minutes,VISp,VISl,VISpm,natural_scenes,free_space_after_GiB
0,732592105,success,0.28,110,40,66,5950,87.3
1,797828357,success,3.42,85,58,52,5950,86.0
2,755434585,success,2.94,75,39,62,5950,83.8
3,756029989,success,3.36,51,30,90,5950,81.4
4,791319847,success,2.87,93,56,17,5950,79.3
